# Figure: one Look-Locker readout step by step

The cell below is tagged `#| label: fig-ll-tip-output`, so `03-look-locker.md` embeds its output with `:::{figure} #fig-ll-tip-output`.

In [ ]:
#| label: fig-ll-tip-output
# Figure: what one Look-Locker readout does to the magnetization, step by step.
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import warnings, logging
# Kaleido (the PNG export for the PDF) can print harmless clean-up warnings on Windows.
warnings.filterwarnings("ignore", message=".*temporary directory.*")
for name in ("choreographer", "kaleido", "logistro"):
    logging.getLogger(name).setLevel(logging.CRITICAL)

# Interactive figure for the website + PNG snapshot for the PDF (same probe as the template).
try:
    pio.to_image(go.Figure(), format="png")
    pio.renderers.default = "plotly_mimetype+png"
except Exception as exc:
    pio.renderers.default = "plotly_mimetype"
    print(f"Static PNG export unavailable ({type(exc).__name__}); PDF will show the caption only.")

# --- Parameters -------------------------------------------------------------
T1 = 1130.0          # muscle at 1.5T [ms], as in the other figures
ALPHA = 20.0         # flip angle [deg], exaggerated so the tip is easy to see
TAU = 200.0          # time between two readouts [ms]
N = 20               # number of readouts (20 x 200 ms = 4 s)
ca, sa = np.cos(np.deg2rad(ALPHA)), np.sin(np.deg2rad(ALPHA))
E = np.exp(-TAU / T1)

def free_recovery(t):
    """Mz without any readouts, after a perfect inversion at t = 0."""
    return 1 - 2 * np.exp(-t / T1)

# --- Simulation: Mz just before each readout ---------------------------------
mz_before = np.empty(N)
mz_before[0] = -1.0                              # perfect inversion
for n in range(1, N):
    mz_before[n] = 1 - (1 - mz_before[n - 1] * ca) * E   # tip, then recover for TAU
t_read = np.arange(N) * TAU

# Two slider positions per readout: (1) just before the pulse, (2) just after the tip
states = []
for n in range(N):
    mz = mz_before[n]
    states.append(dict(t=t_read[n], x=0.0, z=mz, tipped=False, n=n))
    states.append(dict(t=t_read[n], x=mz * sa, z=mz * ca, tipped=True, n=n))

# --- Figure -----------------------------------------------------------------
fig = make_subplots(rows=1, cols=2, column_widths=[0.4, 0.6], horizontal_spacing=0.12)

# Static traces: z-axis guide and the recovery without readouts
fig.add_trace(go.Scatter(x=[0, 0], y=[-1.1, 1.1], mode="lines",
                         line=dict(color="lightgray", width=1), showlegend=False), row=1, col=1)
t_fine = np.linspace(0, t_read[-1], 300)
fig.add_trace(go.Scatter(x=t_fine / 1000, y=free_recovery(t_fine), mode="lines",
                         line=dict(color="lightgray", width=5),
                         name="recovery without readouts"), row=1, col=2)
n_static = 2

TRACES = 6
for i, s in enumerate(states):
    vis = (i == 0)
    ghost = free_recovery(s["t"])
    # left: arrow without readouts (gray) and the real arrow (black)
    fig.add_trace(go.Scatter(x=[0, 0], y=[0, ghost], mode="lines+markers",
                             line=dict(color="lightgray", width=5), marker=dict(size=[0, 10]),
                             name="arrow without readouts", visible=vis), row=1, col=1)
    fig.add_trace(go.Scatter(x=[0, s["x"]], y=[0, s["z"]], mode="lines+markers",
                             line=dict(color="black", width=5), marker=dict(size=[0, 12]),
                             name="magnetization", visible=vis), row=1, col=1)
    # after the tip: the part left along z (blue) and the transverse part = signal (orange)
    fig.add_trace(go.Scatter(x=[0, 0] if s["tipped"] else [None], y=[0, s["z"]] if s["tipped"] else [None],
                             mode="lines", line=dict(color="royalblue", width=3, dash="dot"),
                             name="part left along z (Mz·cos α)", visible=vis), row=1, col=1)
    fig.add_trace(go.Scatter(x=[0, s["x"]] if s["tipped"] else [None], y=[s["z"], s["z"]] if s["tipped"] else [None],
                             mode="lines", line=dict(color="darkorange", width=3),
                             name="transverse part = signal, then spoiled", visible=vis), row=1, col=1)
    # right: points measured so far and the current position
    done = t_read[: s["n"] + (1 if s["tipped"] else 0)]
    fig.add_trace(go.Scatter(x=done / 1000, y=mz_before[: len(done)], mode="markers",
                             marker=dict(size=8, color="black"), name="Mz at each readout",
                             visible=vis), row=1, col=2)
    fig.add_trace(go.Scatter(x=[s["t"] / 1000], y=[s["z"]], mode="markers",
                             marker=dict(size=14, color="crimson", symbol="diamond"),
                             name="now", visible=vis), row=1, col=2)

steps = []
for i, s in enumerate(states):
    visible = [True] * n_static + [False] * (TRACES * len(states))
    visible[n_static + i * TRACES: n_static + (i + 1) * TRACES] = [True] * TRACES
    label = f"{s['t'] / 1000:.1f} s" + (" tip" if s["tipped"] else "")
    steps.append(dict(method="restyle", args=[{"visible": visible}], label=label))

fig.update_layout(
    sliders=[dict(active=0, steps=steps, currentvalue=dict(prefix="Time after inversion: "),
                  pad=dict(t=60))],
    height=680, template="plotly_white",
    # legend at the very bottom of the figure, below the slider, with space in between
    legend=dict(orientation="h", yref="container", yanchor="bottom", y=0.01, x=0),
    margin=dict(t=30, b=230),
)
fig.update_xaxes(title_text="transverse", range=[-1.1, 1.1], zeroline=False, row=1, col=1)
fig.update_yaxes(title_text="Mz / M0", range=[-1.1, 1.1], scaleanchor="x", scaleratio=1, row=1, col=1)
fig.update_xaxes(title_text="Time after inversion [s]", row=1, col=2)
fig.update_yaxes(title_text="Mz / M0", range=[-1.1, 1.1], row=1, col=2)
fig
